In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Optional
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from dataset import Multi30kDataset
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask

from tqdm import tqdm

from nltk.translate.bleu_score import corpus_bleu

import wandb

In [5]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size: int, pad_idx: int = 1, smoothing: float = 0.1):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:

        # logits: (B*T, V)
        # target: (B*T)

        log_probs = F.log_softmax(logits, dim=-1)   # (B * seq, vocab)

        with torch.no_grad():
            # Create smoothed distribution
            true_dist = torch.zeros_like(log_probs)

            # Fill with smoothing value
            true_dist.fill_(self.smoothing / (self.vocab_size - 1))

            # Put confidence at correct indices
            true_dist.scatter(dim = 1, index = target.unsqueeze(1), src = self.confidence)

            # Zero out pad positions
            true_dist[target == self.pad_idx] = 0

        # Compute loss
        loss = -(true_dist * log_probs).sum(dim=1)   # (B * seq)

        # Ignore pad tokens in loss
        non_pad_mask = target != self.pad_idx
        loss = loss[non_pad_mask].mean()

        return loss

In [ ]:
label_smoothing = LabelSmoothingLoss(1024, 1, 0.1)

logits = torch.randn(10, 128, 1024)   
target = torch.randint(0, 128, (10, 128))

logits = logits.view(-1, 1024)
target = target.view(-1)

loss = label_smoothing(logits, target)
print(loss)

In [6]:
logits.shape

torch.Size([1280, 1024])

In [3]:
language_dataset = Multi30kDataset(split = "train")
processed_dataset = language_dataset.process_data()

Building vocab, please wait...


100%|██████████| 29000/29000 [00:00<00:00, 49118.19it/s]


In [3]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return torch.tensor(src), torch.tensor(tgt)
    
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=1, batch_first=True)  # pad_idx = 1
    tgt_batch = pad_sequence(tgt_batch, padding_value=1, batch_first=True)
    return src_batch, tgt_batch

dataset_obj = TranslationDataset(processed_dataset)
dataloader = DataLoader(
    dataset_obj,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

In [5]:
def run_epoch(
    data_iter,
    model: Transformer,
    loss_fn: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
    epoch_num: int = 1,
    is_train: bool = True,
    device: str = "cpu",
) -> float:

    model.train() if is_train else model.eval()
    losses = []
    for _ in range(epoch_num):
        total_loss = 0

        for src, tgt in tqdm(data_iter):

            src = src.to(device)
            tgt = tgt.to(device)

            # masks
            src_mask = make_src_mask(src).to(device)
            tgt_mask = make_tgt_mask(tgt_input).to(device)

            # shift for teacher forcing
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            tgt_mask = make_tgt_mask(tgt_input)

            # forward
            logits = model(src, tgt_input, src_mask, tgt_mask)

            # reshape
            logits = logits.contiguous().view(-1, logits.shape[-1]) # (B * seq, d_model)
            tgt_output = tgt_output.contiguous().view(-1)           # (B * seq, d_model)

            # loss
            loss = loss_fn(logits, tgt_output)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item()
        losses.append(total_loss / len(data_iter))

    return sum(losses)/len(losses)

In [ ]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

label_smoothing = LabelSmoothingLoss(vocab_size = src_vocab, pad_idx = 1, smoothing = 0.1)

optimizer   = optim.Adam(transformer.parameters(), lr=1.0)
scheduler   = NoamScheduler(optimizer, d_model=512, warmup_steps=4000)

loss = run_epoch(data_iter = dataloader, model = transformer, loss_fn = label_smoothing, 
                 optimizer = optimizer, scheduler = scheduler, is_train = True, epoch_num = 1)

In [9]:
for idx in next(iter(dataloader))[0][0]:
    print(language_dataset.en_itos[idx.item()], end = " ")

<sos> while young gate poles while faces walking cafe men <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

In [11]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

src_vocab, tgt_vocab

(18669, 9797)

In [58]:
def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device = "cpu", break_at_eos = True):
    src = src.to(device)
    src_mask = src_mask.to(device)
    
    # encode
    ys = torch.ones(src.size(0), 1).fill_(start_symbol).long().to(device)
    memory = model.encode(src, src_mask)

    # loop
    for _ in range(max_len):
        # decoding
        tgt_mask = make_tgt_mask(ys).to(device)
        out = model.decode(memory, src_mask, ys, tgt_mask)

        prob = out[:, -1, :] # (B, vocab)
        next_word = prob.argmax(dim=-1)

        ys = torch.cat([ys, next_word.unsqueeze(1)], dim=1)

        if (next_word == end_symbol).all() and break_at_eos:
            break
    return ys


sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
sample_src = src[0].unsqueeze(0)
src_mask = make_src_mask(sample_src).to(device)

src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

prediction_idx = greedy_decode(transformer, sample_src, src_mask, max_len = 20, start_symbol = sos_idx, end_symbol = eos_idx, device = "cpu")
prediction_idx

tensor([[   2, 5912, 7782, 7782, 3308, 3997, 6185, 4243, 9639, 5700, 5633, 3738,
         9639, 3997, 7782, 3738, 3106, 6185, 6185, 5633, 3738]])

In [16]:
for idx in prediction_idx[0]:
    print(language_dataset.en_itos[idx.item()], end = " ")

<sos> sandals kaleidoscope kaleidoscope kaleidoscope gateway kaleidoscope fault gateway home kaleidoscope kaleidoscope vulture kaleidoscope kaleidoscope trapeze kaleidoscope fault kaleidoscope kaleidoscope fault 

In [61]:
def evaluate_bleu(
    model: Transformer,
    test_dataloader,
    tgt_vocab,
    device: str = "cpu",
    max_len: int = 100,
) -> float:

    model.eval()

    references = []
    hypotheses = []

    token_to_idx = {v: i for i, v in tgt_vocab.items()}
    sos_idx = token_to_idx["<sos>"]
    eos_idx = token_to_idx["<eos>"]
    pad_idx = token_to_idx["<pad>"]

    itos = tgt_vocab  # idx -> token

    with torch.no_grad():
        for num, (src, tgt) in enumerate(test_dataloader):
            # if num == 5: break
            B = src.size(0)

            ys = greedy_decode(model = model, src = src, 
                          src_mask = make_src_mask(src).to(device),
                          max_len = max_len, start_symbol = sos_idx, end_symbol = eos_idx,
                          device = device, break_at_eos = False 
                          )

            # Convert predictions and targets to words
            for i in tqdm(range(B)):

                pred_tokens = ys[i].tolist()
                tgt_tokens  = tgt[i].tolist()

                # Remove special tokens
                pred_sentence = []
                for tok in pred_tokens:
                    if tok == eos_idx:
                        break
                    if tok not in [sos_idx, pad_idx]:
                        pred_sentence.append(itos[tok])

                ref_sentence = []
                for tok in tgt_tokens:
                    if tok == eos_idx:
                        break
                    if tok not in [sos_idx, pad_idx]:
                        ref_sentence.append(itos[tok])

                hypotheses.append(pred_sentence)
                references.append([ref_sentence])  # list of references

    bleu = corpus_bleu(references, hypotheses) * 100
    return bleu

In [ ]:
sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
sample_src = src[0].unsqueeze(0)
src_mask = make_src_mask(sample_src).to(device)

src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

evaluate_bleu(transformer, dataloader, language_dataset.en_itos)

In [ ]:
def save_checkpoint(
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    scheduler,
    epoch: int,
    path: str = "checkpoint.pt",
) -> None:
    torch.save(
        {
            "epoch"               : epoch,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_config": {
                "src_vocab_size": model.src_vocab_size,
                "tgt_vocab_size": model.tgt_vocab_size,
                "d_model"       : model.d_model,
                "N"             : model.N,
                "num_heads"     : model.num_heads,
                "d_ff"          : model.d_ff,
                "dropout"       : model.dropout,
            }
         }
    ,path
    )

def load_checkpoint(
    path: str,
    model: Transformer,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
    device = "cpu"
) -> int:

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    return checkpoint["epoch"]

In [ ]:
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=1, batch_first=True)  # pad_idx = 1
    tgt_batch = pad_sequence(tgt_batch, padding_value=1, batch_first=True)
    return src_batch, tgt_batch


In [ ]:
# 2. Build dataset / vocabs from dataset.py
language_dataset = Multi30kDataset(split = "train")

config = {
    "src_vocab_size"   : len(language_dataset.de_vocab),
    "tgt_vocab_size"   : len(language_dataset.en_vocab),
    "d_model"          : 512,
    "N"                : 6,
    "num_heads"        : 8,
    "d_ff"             : 2048,
    "dropout"          : 0.1,
    "train_batch_size" : 32,
    "test_batch_size"  : 32,
    "epochs"           : 10,
    "device"           : 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_every'       : 4
}


# 1. Init W&B
wandb.init(project="Machine Translation Transformer", config = config)

# 3. Create DataLoaders for train / val 
processed_dataset = language_dataset.process_data()
train_dataset_obj = TranslationDataset(processed_dataset)
train_dataloader = DataLoader(
    train_dataset_obj,
    batch_size=config["train_batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)

language_dataset.split = "test"
processed_dataset = language_dataset.process_data()
test_dataset_obj = TranslationDataset(processed_dataset)
test_dataloader = DataLoader(
    test_dataset_obj,
    batch_size=config["test_batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)

# 4. Instantiate Transformer with hyperparameters from config
transformer = Transformer(src_vocab_size = config["src_vocab_size"], tgt_vocab_size = config['tgt_vocab_size'],
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1).to(config["device"])

# 5. Instantiate Adam optimizer (β1=0.9, β2=0.98, ε=1e-9)
optimizer = optim.Adam(transformer.parameters(), betas = [0.9, 0.98], lr=1e-4)

# 6. Instantiate NoamScheduler(optimizer, d_model, warmup_steps=4000)
scheduler = NoamScheduler(optimizer, d_model = config["d_model"], warmup_steps = 4000)

# 7. Instantiate LabelSmoothingLoss(vocab_size, pad_idx, smoothing=0.1)
loss_fn = LabelSmoothingLoss(vocab_size = config["tgt_vocab_size"], pad_idx = 1, smoothing = 0.1)

# 8. Training loop:
for epoch in range(config['epochs']):
    transformer.train()
    train_loss = run_epoch(train_dataloader, transformer, loss_fn,
                    optimizer, scheduler, epoch, is_train=True, epoch_num = 1)
    transformer.eval()
    test_loss = run_epoch(test_dataloader, transformer, loss_fn,
                    None, None, epoch, is_train=False, epoch_num = 1)
    wandb.log({'epoch': epoch, 'train_loss': train_loss, 'test_loss': test_loss})
    
    if epoch % config['save_every'] == 0:
        save_checkpoint(transformer, optimizer, scheduler, epoch)

# 9. Final BLEU on test set
bleu = evaluate_bleu(transformer, test_dataloader, language_dataset.en_itos, device = config['device'])
wandb.log({'test_bleu': bleu})

In [ ]:
language_dataset.split = "test"

'train'

In [1]:
import torch
from torch import nn

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)                                  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # positions: (max_len, 1)
        div_term = 1 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))

        # apply sin to even indices
        pe[:, 0::2] = torch.sin(position * div_term)

        # apply cos to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)

        # reshape to (1, max_len, d_model) for broadcasting
        pe = pe.unsqueeze(0)

        # register as buffer (not a parameter)
        self.register_buffer("pe", pe)

        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.shape[1]
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)
        
    
pe = PositionalEncoding(512, 0.1, max_len=1000)
pe(torch.rand(5, 182, 512)).shape

torch.Size([5, 182, 512])

In [31]:
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
 
        self.embedding = torch.nn.Embedding(max_len, d_model)

    def forward(self, x:torch.tensor):
        return self.dropout(self.embedding(x))


In [34]:
lpe = LearnedPositionalEncoding(512, 0.1, 1000)
lpe(torch.randint(5, 512, (12, 10, 512))).shape

torch.Size([12, 10, 512, 512])

In [12]:
transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

In [8]:
language_dataset = Multi30kDataset(split = "train")

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
config = {
    "src_vocab_size"   : len(language_dataset.de_vocab),
    "tgt_vocab_size"   : len(language_dataset.en_vocab),
    "d_model"          : 512,
    "N"                : 6,
    "num_heads"        : 8,
    "d_ff"             : 2048,
    "dropout"          : 0.1,
    "train_batch_size" : 32,
    "test_batch_size"  : 32,
    "epochs"           : 1,
    "device"           : device,
    'save_every'       : 4
}



# 3. Create DataLoaders for train / val 
processed_dataset = language_dataset.process_data()
train_dataset_obj = TranslationDataset(processed_dataset)
train_dataloader = DataLoader(
    train_dataset_obj,
    batch_size=config["train_batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)

language_dataset.split = "test"
processed_dataset = language_dataset.process_data()
test_dataset_obj = TranslationDataset(processed_dataset)
test_dataloader = DataLoader(
    test_dataset_obj,
    batch_size=config["test_batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)

# 4. Instantiate Transformer with hyperparameters from config
transformer = Transformer(src_vocab_size = config["src_vocab_size"], tgt_vocab_size = config['tgt_vocab_size'],
                        d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                        dropout = 0.1).to(config["device"])

# 5. Instantiate Adam optimizer (β1=0.9, β2=0.98, ε=1e-9)
optimizer = optim.Adam(transformer.parameters(), betas = [0.9, 0.98], lr=1)

# 6. Instantiate NoamScheduler(optimizer, d_model, warmup_steps=4000)
scheduler = NoamScheduler(optimizer, d_model = config["d_model"], warmup_steps = 5000)

# 7. Instantiate LabelSmoothingLoss(vocab_size, pad_idx, smoothing=0.1)
loss_fn = LabelSmoothingLoss(vocab_size = config["tgt_vocab_size"], pad_idx = 1, smoothing = 0.1)

# 8. Training loop:
# for epoch in range(config['epochs']):
transformer.train()
losses = []

total_loss = 0
b = 1
for src, tgt in tqdm(train_dataloader):

    src = src.to(device)
    tgt = tgt.to(device)

    # masks
    src_mask = make_src_mask(src).to(device)
    tgt_mask = make_tgt_mask(tgt).to(device)

    # shift for teacher forcing
    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]

    tgt_mask = make_tgt_mask(tgt_input)

    # forward
    logits = transformer(src, tgt_input, src_mask, tgt_mask)

    # reshape
    logits = logits.contiguous().view(-1, logits.shape[-1]) # (B * seq, d_model)
    tgt_output = tgt_output.contiguous().view(-1)           # (B * seq, d_model)

    # loss
    loss = loss_fn(logits, tgt_output)

    
    optimizer.zero_grad()
    loss.backward()
    break
    optimizer.step()

    if scheduler is not None:
        scheduler.step()

    

Building vocab, please wait...


  0%|          | 0/907 [00:00<?, ?it/s]


TypeError: scatter() received an invalid combination of arguments - got (src=float, index=Tensor, dim=int, ), but expected one of:
 * (int dim, Tensor index, Tensor src)
      didn't match because some of the arguments have invalid types: (dim=int, index=Tensor, !src=float!, )
 * (int dim, Tensor index, Tensor src, *, str reduce)
 * (name dim, Tensor index, Tensor src)
      didn't match because some of the arguments have invalid types: (!dim=int!, index=Tensor, !src=float!, )
 * (int dim, Tensor index, Number value)
      didn't match because some of the keywords were incorrect: src
 * (int dim, Tensor index, Number value, *, str reduce)
 * (name dim, Tensor index, Number value)
      didn't match because some of the keywords were incorrect: src


In [3]:
q_norm, k_norm, v_norm = None, None, None

for name, param in transformer.named_parameters():
    if param.grad is None:
        continue

    grad_norm = param.grad.detach().norm(2).item()

    if "WQ" in name:
        q_norm = grad_norm

    elif "WK" in name:
        k_norm = grad_norm

    elif "WV" in name:
        v_norm = grad_norm

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x31ce2ee50>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 10773c750, raw_cell="q_norm, k_norm, v_norm = None, None, None

for nam.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/mohamedmafaz/Desktop/Machine%20Translation%20Transformer/train.ipynb#X33sZmlsZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

NameError: name 'transformer' is not defined

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x31ce2ee50>> (for post_run_cell), with arguments args (<ExecutionResult object at 317dba150, execution_count=3 error_before_exec=None error_in_exec=name 'transformer' is not defined info=<ExecutionInfo object at 10773c750, raw_cell="q_norm, k_norm, v_norm = None, None, None

for nam.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/mohamedmafaz/Desktop/Machine%20Translation%20Transformer/train.ipynb#X33sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

Building vocab, please wait...


Processing test: 100%|██████████| 1000/1000 [00:00<00:00, 44499.54it/s]
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mohdmafaz200303 (mafaz03). Use `wandb login --relogin` to force relogin


  0%|          | 1/907 [00:11<2:48:03, 11.13s/it]2026-05-17 11:08:55.359 Python[24624:9253975] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-24624-2026-05-17_11_08_54-4241710595‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
  0%|          | 1/907 [00:16<4:09:05, 16.50s/it]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x333dfe610>> (for post_run_cell), with arguments args (<ExecutionResult object at 32bd78c10, execution_count=58 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 333e20850, raw_cell="from train import run_training_experiment
run_trai.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/mohamedmafaz/Desktop/Machine%20Translation%20Transformer/train.ipynb#X40sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [55]:
q_norm